In [1]:
from itertools import permutations
from z3 import *

web_portal, add_services, site_stats, basic, advanced, site_search, images, text, html, dynamic, \
web_server, logging, db, file, protocol, http, ftp, https, cont, static, active, php, jsp, cgi, \
performance, ms, sec = Bools(
    "web_portal add_services site_stats basic advanced site_search images text html dynamic "
    "web_server logging db file protocol http ftp https cont static active php jsp cgi "
    "performance ms sec"
)

features = [
    web_portal, add_services, site_stats, basic, advanced, site_search, images, text, html, dynamic,
    web_server, logging, db, file, protocol, http, ftp, https, cont, static, active, php, jsp, cgi,
    performance, ms, sec
]

def fazerSolver():

    s = Solver()

    # -------------------------------------------------------------------------
    # 2. Restrições do Feature Model
    # -------------------------------------------------------------------------

    # web_portal é a raiz (root feature)[cite: 1]
    s.add(web_portal)

    # --- Filhos diretos de web_portal ---

    # add_services é opcional em web_portal[cite: 1, 2]
    s.add(Implies(add_services, web_portal))


    # web_server é obrigatório em web_portal[cite: 1, 2]
    s.add(web_server == web_portal)


    # performance é opcional em web_portal[cite: 1, 2]
    s.add(Implies(performance, web_portal))

    # --- Subárvore: add_services ---
    # site_stats e site_search são opcionais em add_services[cite: 1, 2]
    s.add(Implies(site_stats, add_services))
    s.add(Implies(site_search, add_services))

    # basic é obrigatório e advanced é opcional em site_stats[cite: 1, 2]
    s.add(basic == site_stats)
    s.add(Implies(advanced, site_stats))

    # images e text são opcionais em site_search[cite: 1, 2]
    s.add(Implies(images, site_search))
    s.add(Implies(text, site_search))

    # html é obrigatório e dynamic é opcional em text[cite: 1, 2]
    s.add(html == text)
    s.add(Implies(dynamic, text))

    # --- Subárvore: web_server ---
    # logging e protocol são opcionais em web_server[cite: 1, 2]
    s.add(Implies(logging, web_server))
    s.add(Implies(protocol, web_server))
    # cont é obrigatório em web_server[cite: 1, 2]
    s.add(cont == web_server)

    # db e file são sub-features XOR (Alternative) de logging[cite: 1, 2, 3]
    s.add(Or(db, file) == logging)
    s.add(Not(And(db, file)))

    # http, ftp e https são sub-features OR de protocol[cite: 1, 2, 3]
    s.add(Or(http, ftp, https) == protocol)

    # static é obrigatório e active é opcional em cont[cite: 1, 2]
    s.add(static == cont)
    s.add(Implies(active, cont))

    # php, jsp e cgi são sub-features OR de active[cite: 1, 2, 3]
    s.add(Or(php, jsp, cgi) == active)

    # --- Subárvore: performance ---
    # ms e sec são sub-features XOR (Alternative) de performance[cite: 1, 2, 3]
    s.add(Or(ms, sec) == performance)
    s.add(Not(And(ms, sec)))

    # -------------------------------------------------------------------------
    # 3. Cross-Tree Constraints (Setas tracejadas)
    # -------------------------------------------------------------------------
    # file requires ftp[cite: 1, 2, 3]
    s.add(Implies(file, ftp))

    # https excludes ms
    s.add(Not(And(https, ms)))

    # dynamic requires active[cite: 1, 2, 3]
    s.add(Implies(dynamic, active))
    return s

s = fazerSolver()

if s.check() == sat:
  print("Feature model is non void.")
else:
  print("Feature model is void.")

Feature model is non void.


In [2]:
if s.check() == sat:
  m = s.model()
  print(m)

[advanced = False,
 dynamic = False,
 site_stats = False,
 site_search = False,
 performance = False,
 static = True,
 db = False,
 html = False,
 logging = False,
 ftp = False,
 add_services = False,
 https = False,
 protocol = False,
 cont = True,
 ms = False,
 images = False,
 file = False,
 basic = False,
 text = False,
 active = False,
 php = False,
 cgi = False,
 sec = False,
 web_portal = True,
 http = False,
 jsp = False,
 web_server = True]


In [3]:


for f in features:
  s.push()
  s.add(f)
  if s.check() == unsat:
    print("Feature " + f.decl().name() + "is dead!")
  s.pop()

In [4]:
for f in features:
  s.push()
  s.add(Not(f))
  if s.check() == unsat:
    print("Feature " + f.decl().name() + " is core!")

  s.pop()

Feature web_portal is core!
Feature web_server is core!
Feature cont is core!
Feature static is core!


In [5]:
s.push()
i = 0
res = 0
while s.check() == sat:
  i += 1
  m = s.model()
  p = []
  c = 0
  for f in features:
    if is_true(m[f]):
      p.append(f)
      c += 1
      #print(f.decl().name(),end=" ")
    else:
      p.append(Not(f))
    if c > 11:
      break
    
  if c == 11:
    res += 1
  #print()
  s.add(Not(And(p)))
#print()
print("There are " + str(i) + " possible products!")
print("There are " + str(res) + " possible products com 11!")
s.pop()

There are 1396 possible products!
There are 282 possible products com 11!


In [6]:
s.push()
i = 0
perm = permutations(features, 2)
while s.check() == sat:
  i += 1
  m = s.model()
  p = []
  for f in features:
    if is_true(m[f]):
      p.append(f)
      #print(f.decl().name(),end=" ")
    else:
      p.append(Not(f))
  #print()
  s.add(Not(And(p)))
#print()
print("There are " + str(i) + " possible products!")
s.pop()

There are 8500 possible products!


In [7]:
perm = permutations(features, 2)
#list(perm)

In [ ]:
s = fazerSolver()

s.push()
perm = permutations(features, 2)
resN = 0
res = None
lres = []
for (f1,f2) in perm:
    s.push()
    s.add(Implies(f1,f2))

    if s.check() == unsat:
        s.pop()

        continue

    l = []
    c = 0
    for f in features:
        s.push()
        s.add(f)
        if s.check() == unsat:
            l.append(f)
            c += 1
            #print("Feature " + f.decl().name() + "is dead!")
        s.pop()
    if resN < c:
        resN = c
        res = (f1,f2)
        lres = l
    s.pop()


print("feats: " + str(res) + " nr: " + str(resN) + " " + str(lres))

s.pop()
    

feats: (web_portal, ms) nr: 2 [https, sec]
